# Cómo elige palabras un LLM: temperatura, top-k y top-p

**Facsímil 3 · Arquitecturas y modelos** — capítulo 4 (MLP, residual, *logits* y *sampling*).

Aquí desmontamos una de las confusiones más comunes sobre los LLM: la idea de que el modelo «elige» la
siguiente palabra. No la elige. El modelo produce una **puntuación** (un *logit*) para *cada* palabra de
su vocabulario, y un proceso de **muestreo** —que tú controlas con tres botones— decide cuál sale. En
este cuaderno te pones en la piel de esa última capa y ves cómo la **temperatura**, el **top-k** y el
**top-p** transforman al mismo modelo de un loro predecible a un poeta disparatado. Entender estos
botones es dejar de sufrir respuestas «raras» y empezar a controlarlas.

### Qué vas a aprender
- Qué son los **logits** y cómo la **softmax** los convierte en una distribución de probabilidad.
- Cómo la **temperatura** modela el atrevimiento del modelo, con su fórmula y su intuición.
- Qué hacen **top-k** y **top-p** (núcleo) y en qué se diferencian, con un caso donde uno falla.
- A medir la **incertidumbre** (entropía) y a *ver* la diferencia entre estrategias muestreando miles de
  veces.
- Por qué la combinación temperatura + top-p es la receta de serie de los asistentes.
- Qué es la **penalización por repetición** y por qué evita que el modelo se atasque.

### Cuánto cuesta
Unos 18 minutos. CPU, sin claves. Lee cada celda de texto antes de correr la de código.


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

# Despues de "El cafe de la manana me ayuda a ___", estas son las puntuaciones
# (logits) que daria el modelo a algunas continuaciones. Las ponemos a mano para
# ver la mecanica; en un LLM real salen de la red (capitulos 2-4).
vocab  = ["despertar", "concentrarme", "trabajar", "vivir", "pensar", "volar", "programar", "bailar"]
logits = np.array([3.2, 2.8, 2.5, 1.4, 1.2, -0.5, 0.8, -1.0])

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

p = softmax(logits)
print("Distribucion 'cruda' (softmax de los logits):")
for w, pi in sorted(zip(vocab, p), key=lambda t: -t[1]):
    print(f"  {w:>14}: {pi*100:5.1f}%  {'#'*int(pi*40)}")
print(f"\nSuma de todas las probabilidades: {p.sum():.4f} (tiene que ser 1).")


## 1. De logits a probabilidades: la softmax

La última capa del modelo no da probabilidades, da **logits**: números reales sin acotar, uno por cada
palabra del vocabulario (que en un LLM real tiene decenas de miles de entradas). Para convertirlos en
una distribución de probabilidad —números entre 0 y 1 que suman 1— se aplica la **softmax**:
$$ p_i = \frac{e^{z_i}}{\sum_j e^{z_j}} $$
La exponencial agranda las diferencias (lo más probable se separa de lo improbable) y la división
normaliza. El resultado es la distribución que ves arriba: el modelo «cree» que «despertar» es lo más
probable, pero deja una rendija a las demás. Lo que viene ahora es **moldear** esa distribución.


### Un detalle que confunde: la softmax solo mira diferencias

La softmax no se fija en el valor absoluto de los *logits*, solo en sus **diferencias**. Si a todos los
*logits* les sumas la misma constante, la distribución no cambia un ápice. Por eso en el código restamos
`z.max()` antes de exponenciar: no altera el resultado y evita que `exp` desborde con números grandes.
Lo comprobamos.


In [ ]:
desplazados = logits + 100.0     # sumamos 100 a TODOS los logits
print("Probabilidad de 'despertar' con logits normales :", f"{softmax(logits)[0]*100:.2f}%")
print("Probabilidad de 'despertar' sumando 100 a todos  :", f"{softmax(desplazados)[0]*100:.2f}%")
print("Identicas:", np.allclose(softmax(logits), softmax(desplazados)))
print("\nMoraleja: a la softmax solo le importa CUANTO mas alto es un logit que otro, no su valor suelto.")


## 2. La temperatura: subir o bajar el atrevimiento

La temperatura $T$ divide los *logits* antes de la softmax: $p_i \propto e^{z_i / T}$. La intuición
viene de la física (de ahí el nombre): a temperatura **baja**, el sistema se «congela» en su estado más
probable; a temperatura **alta**, se agita y visita estados improbables.

- $T < 1$ (por ejemplo 0.5): **agranda** las diferencias. El modelo se vuelve seguro, predecible,
  repetitivo. Ideal para extraer datos o clasificar.
- $T = 1$: la distribución original.
- $T > 1$ (por ejemplo 1.5): **aplana** las diferencias. Da oportunidades a palabras improbables: aparece
  la creatividad... y los disparates.


In [ ]:
print(f"{'palabra':>14} | T=0.5    T=1.0    T=1.5")
for i, w in enumerate(vocab):
    fila = "   ".join(f"{softmax(logits/T)[i]*100:4.0f}%" for T in [0.5, 1.0, 1.5])
    print(f"{w:>14} |  {fila}")
print("\nT baja concentra en 'despertar'; T alta reparte y da aire incluso a 'volar' o 'bailar'.")


### Verlo como curvas: la misma distribución, tres atrevimientos

La tabla está bien, pero un dibujo lo deja claro de un vistazo. Pintamos la distribución entera a tres
temperaturas. Verás cómo la temperatura baja levanta un pico y aplasta el resto, mientras que la alta
allana el terreno hacia un reparto más parejo.


In [ ]:
x = np.arange(len(vocab))
plt.figure(figsize=(8, 3.6))
for T, marca in [(0.5, "o-"), (1.0, "s-"), (1.5, "^-")]:
    plt.plot(x, softmax(logits/T)*100, marca, label=f"T = {T}")
plt.xticks(x, vocab, rotation=45, ha="right")
plt.ylabel("probabilidad (%)"); plt.title("La temperatura cambia la forma de la distribucion")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print("T=0.5: un pico afilado en 'despertar'. T=1.5: una loma suave que reparte entre muchas.")


## 3. Medir la incertidumbre: la entropía

¿Cómo ponemos número a «cómo de repartida» está una distribución? Con la **entropía**: $H = -\sum_i p_i
\log_2 p_i$. Vale 0 cuando el modelo está completamente seguro (toda la probabilidad en una palabra) y
crece cuanto más repartida está (máxima cuando todas son igual de probables). La temperatura es, en el
fondo, un mando de entropía: subirla aumenta la incertidumbre de la siguiente palabra.


In [ ]:
def entropia(p):
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

print("temperatura | entropia (bits) | interpretacion")
for T in [0.2, 0.5, 1.0, 1.5, 3.0]:
    h = entropia(softmax(logits/T))
    etq = 'casi decidido' if h < 1 else 'algo de variedad' if h < 2 else 'muy abierto'
    print(f"    {T:>4}    |     {h:4.2f}      | {etq}")
print(f"\nMaxima entropia posible con {len(vocab)} palabras: {np.log2(len(vocab)):.2f} bits (todas iguales).")


### La curva del atrevimiento

Si recorremos muchas temperaturas y dibujamos la entropía, sale la «curva del atrevimiento» del modelo:
sube de forma suave desde casi 0 (todo a una palabra) hacia el máximo teórico (todas por igual). Es la
misma información que la tabla, pero continua. La línea de puntos marca el techo: con 8 palabras no se
puede pasar de 3 bits.


In [ ]:
Ts = np.linspace(0.1, 5.0, 40)
Hs = [entropia(softmax(logits/T)) for T in Ts]
plt.figure(figsize=(7, 3.6))
plt.plot(Ts, Hs, "-")
plt.axhline(np.log2(len(vocab)), ls="--", color="gray", label="maximo (todas iguales)")
plt.xlabel("temperatura"); plt.ylabel("entropia (bits)")
plt.title("Mas temperatura, mas incertidumbre"); plt.legend()
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()
print(f"A T=0.1 la entropia es {Hs[0]:.2f} bits (casi decidido); a T=5 sube a {Hs[-1]:.2f} bits (muy abierto).")


## 4. Top-k y top-p: cortar la cola de los disparates

La temperatura cambia la *forma* de toda la distribución. Otra familia de técnicas, en cambio,
**recorta**: deja fuera las opciones malas para que nunca se muestreen, por improbables que sean tras
subir la temperatura.

- **Top-k:** se queda solo con las `k` palabras más probables y reparte entre ellas. Simple, pero `k`
  es fijo: a veces corta de más, a veces de menos.
- **Top-p (núcleo):** se queda con las palabras que **acumulan** el `p`% de probabilidad. Es
  adaptativo: cuando el modelo lo tiene claro, mantiene pocas; cuando duda, mantiene muchas. Suele dar
  mejores resultados que top-k.


In [ ]:
def top_k(p, k):
    q = p.copy(); q[np.argsort(p)[:-k]] = 0; return q / q.sum()
def top_p(p, umbral):
    orden = np.argsort(p)[::-1]; acum = np.cumsum(p[orden])
    mantener = orden[:np.searchsorted(acum, umbral) + 1]
    q = np.zeros_like(p); q[mantener] = p[mantener]; return q / q.sum()

print("Top-k=3 (solo las 3 mejores; el resto, prohibido):")
for w, pi in sorted(zip(vocab, top_k(p, 3)), key=lambda t: -t[1]):
    if pi > 0: print(f"  {w:>14}: {pi*100:5.1f}%")
print("\nTop-p=0.9 (las que juntas suman el 90%):")
mantenidas = sum(1 for pi in top_p(p, 0.9) if pi > 0)
for w, pi in sorted(zip(vocab, top_p(p, 0.9)), key=lambda t: -t[1]):
    if pi > 0: print(f"  {w:>14}: {pi*100:5.1f}%")
print(f"  -> top-p mantuvo {mantenidas} palabras (las justas para sumar el 90%).")


### El caso donde top-k se queda corto (y top-p acierta)

`k` es un número fijo, y ahí está su talón de Aquiles. Imagina dos situaciones: una en la que el modelo
lo tiene **clarísimo** (una palabra se lleva casi todo) y otra en la que **duda** (la probabilidad está
muy repartida). Con un `k` fijo de 3, en la primera situación dejas entrar palabras casi imposibles, y
en la segunda cortas opciones razonables. Top-p, al mirar la probabilidad **acumulada**, se adapta solo.
Lo vemos con dos distribuciones distintas.


In [ ]:
seguro  = softmax(np.array([6.0, 1.0, 0.5, 0.2, 0.0, -0.5, -1.0, -1.5]))   # un claro favorito
dudoso  = softmax(np.array([1.2, 1.1, 1.0, 0.9, 0.8, 0.7, 0.6, 0.5]))        # casi empate

for nombre, dist in [("modelo SEGURO", seguro), ("modelo DUDOSO", dudoso)]:
    n_k = int((top_k(dist, 3) > 0).sum())
    n_p = int((top_p(dist, 0.9) > 0).sum())
    print(f"{nombre:>14}: top-k=3 deja {n_k} palabras | top-p=0.9 deja {n_p} palabras")
print("\nTop-k siempre deja 3, dude o no el modelo. Top-p deja pocas cuando hay un favorito claro")
print("y muchas cuando el modelo duda: se ajusta a la confianza real de cada paso.")


### El umbral de top-p, escalón a escalón

Para coger intuición del «núcleo», barremos el umbral `p` de bajo a alto sobre nuestra distribución y
vemos cuántas palabras deja entrar y cuánta probabilidad real cubren. Con un umbral pequeño solo
sobrevive el favorito; al subirlo, el núcleo va admitiendo palabras hasta abrazar casi todo el
vocabulario. Esa es la idea de quedarse con «la masa principal» de la probabilidad.


In [ ]:
print("umbral p | palabras en el nucleo | probabilidad real que cubren")
for umbral in [0.3, 0.5, 0.7, 0.9, 0.99]:
    q = top_p(p, umbral)
    cuantas = int((q > 0).sum())
    cubre = p[q > 0].sum()        # probabilidad ORIGINAL de las palabras mantenidas
    print(f"  {umbral:>5}   |        {cuantas:>2}            |        {cubre*100:5.1f}%")
print("\nMas umbral, mas palabras: el nucleo crece de forma adaptativa hasta cubrir lo que le pidas.")


## 5. Lo que de verdad cambia: generar muchas veces

Los porcentajes son abstractos. Hagámoslo tangible: muestreamos 5000 veces la siguiente palabra con
cada estrategia y contamos. Así *ves* la diferencia entre «siempre lo más probable» (*greedy*, que es
el límite $T \to 0$), «con algo de variedad» y «barra libre».


In [ ]:
def muestrea(p, n=5000):
    idx = np.random.choice(len(p), size=n, p=p)
    cuenta = {vocab[i]: int((idx == i).sum()) for i in range(len(p)) if (idx == i).sum() > 0}
    return dict(sorted(cuenta.items(), key=lambda t: -t[1]))

print("Greedy (T->0): SIEMPRE 'despertar' (la mas probable, sin variedad).")
print("T=1.0    :", muestrea(p))
print("T=1.5    :", muestrea(softmax(logits/1.5)))
print("Top-p 0.9:", muestrea(top_p(p, 0.9)))


### Cuántas palabras distintas salen, en realidad

Otra forma de ver «cuánto explora» cada estrategia: contar **cuántas palabras distintas** aparecen al
menos una vez en las 5000 tiradas. Greedy produce 1 (siempre la misma); top-p un puñado (recorta la
cola); temperatura alta casi todas (no recorta nada). El número resume de un plumazo el equilibrio entre
fiabilidad y exploración.


In [ ]:
for nombre, dist in [("T=0.5", softmax(logits/0.5)), ("T=1.0", p),
                      ("T=1.5", softmax(logits/1.5)), ("top-p 0.9", top_p(p, 0.9)),
                      ("top-k 3", top_k(p, 3))]:
    distintas = len(muestrea(dist))
    print(f"  {nombre:>9}: {distintas} palabras distintas en 5000 tiradas  {'*'*distintas}")
print("\nMenos palabras distintas = mas fiable y repetitivo. Mas = mas variado (y mas arriesgado).")


**La lección.** Con *greedy* o temperatura baja, el modelo dice casi siempre lo mismo: fiable para
extraer datos, clasificar o seguir instrucciones al pie de la letra. Con temperatura alta o *top-p*
amplio, explora: bueno para escribir, malo para precisión. No hay un valor «correcto»: depende de si
quieres un **notario** o un **poeta**. Y ese botón es tuyo, no del modelo. Muchas «alucinaciones
creativas» y muchas «respuestas planas» se explican (y se domestican) justo aquí.


## 6. La receta real: primero recortar, luego muestrear

Los asistentes no usan estos botones por separado: los **encadenan**. La receta habitual es *top-p*
para quitar la cola de disparates **y luego** temperatura para dar variedad entre lo que queda. Así
nunca sale una palabra absurda (la cola está cortada), pero sí hay margen creativo entre las opciones
sensatas. Lo montamos: aplicamos top-p y, sobre lo que sobrevive, subimos la temperatura.


In [ ]:
def top_p_y_temperatura(logits, umbral, T):
    p0 = softmax(logits)
    orden = np.argsort(p0)[::-1]; acum = np.cumsum(p0[orden])
    mantener = orden[:np.searchsorted(acum, umbral) + 1]
    z = np.full_like(logits, -np.inf)
    z[mantener] = logits[mantener]          # solo sobreviven los del nucleo
    return softmax(z / T)

receta = top_p_y_temperatura(logits, umbral=0.9, T=1.3)
print("Top-p 0.9 y luego T=1.3 (la receta de muchos asistentes):")
for w, pi in sorted(zip(vocab, receta), key=lambda t: -t[1]):
    if pi > 0: print(f"  {w:>14}: {pi*100:5.1f}%")
fuera = [w for w, pi in zip(vocab, receta) if pi == 0]
print(f"\nFuera de juego (la cola recortada): {fuera}")
print("Variedad entre las sensatas, cero opciones absurdas: lo mejor de los dos mundos.")


## 7. El bucle se atasca: penalización por repetición

Hay un fallo clásico de la generación *greedy* o de temperatura baja: el modelo entra en **bucle** y
repite la misma palabra o frase una y otra vez («... y entonces y entonces y entonces...»). Una defensa
sencilla es **penalizar** los *tokens* ya usados: restar un poco a su *logit* cada vez que aparecen, para
que pierdan atractivo. Lo simulamos generando una secuencia con y sin penalización.


In [ ]:
def genera(logits_base, pasos=12, penalizacion=0.0, T=0.7, semilla=3):
    rng = np.random.default_rng(semilla)
    usados = np.zeros(len(logits_base))
    salida = []
    for _ in range(pasos):
        z = logits_base - penalizacion * usados      # castiga lo ya dicho
        prob = softmax(z / T)
        i = rng.choice(len(prob), p=prob)
        salida.append(vocab[i]); usados[i] += 1
    return salida

sin_pen = genera(logits, penalizacion=0.0)
con_pen = genera(logits, penalizacion=1.5)
print("Sin penalizacion:", " ".join(sin_pen))
print(f"  -> palabras distintas: {len(set(sin_pen))} de {len(sin_pen)}")
print("Con penalizacion 1.5:", " ".join(con_pen))
print(f"  -> palabras distintas: {len(set(con_pen))} de {len(con_pen)}")
print("\nLa penalizacion empuja al modelo a no repetirse: mas variedad, menos bucles.")


## 8. Pruébalo tú

1. **Pon `T = 0.1`** en la tabla de temperaturas: ¿queda alguna palabra con opciones además de
   «despertar»? Eso es casi *greedy*.
2. **Cambia el umbral de top-p** a 0.5 y a 0.99: ¿cuántas palabras mantiene en cada caso?
3. **Empata dos opciones:** cambia los *logits* para que «trabajar» y «despertar» estén casi iguales.
   ¿Cómo afecta eso a *top-p* (que se adapta) frente a *top-k* (que es fijo)?
4. **Sube la penalización por repetición** a 3.0 y bájala a 0.0: ¿cómo cambia la variedad de la secuencia
   generada?
5. **Combina a tu gusto:** en la receta del apartado 6, prueba `umbral=0.6, T=1.8`. ¿Qué palabras quedan
   dentro del núcleo?
6. **Dibuja tu propia curva de entropía** cambiando los *logits* de partida: ¿una distribución más plana
   parte de más entropía?


## 9. Errores comunes

- **Creer que el modelo «decide» la palabra.** Da una distribución; el muestreo decide. Cambiar los
  parámetros de muestreo cambia la salida sin tocar el modelo.
- **Usar temperatura alta para tareas de precisión** (extraer un dato, clasificar). Ahí quieres
  *greedy* o `T` baja: fiabilidad, no creatividad.
- **Confundir top-k y top-p.** `k` es fijo; `p` se adapta a la confianza del modelo. Lo has visto: top-k
  deja siempre el mismo número de palabras, dude el modelo o no.
- **Olvidar que estos botones interactúan.** Temperatura, top-k y top-p se combinan; el efecto final es
  de los tres juntos, no de uno solo.
- **Ignorar la repetición.** Sin penalización, temperatura baja puede meter al modelo en un bucle. No es
  un fallo del modelo, es del muestreo.


## 10. Qué te llevas

- Un LLM produce **una distribución sobre todo el vocabulario**; el texto sale de **muestrear** esa
  distribución, no de una verdad única.
- La **temperatura** moldea la forma (atrevimiento, medible como entropía); **top-k/top-p** recortan la
  cola (seguridad). Juntos, más la **penalización por repetición**, controlan el equilibrio entre
  fiabilidad, creatividad y no atascarse.
- La receta real **encadena**: recortar con top-p y luego dar variedad con temperatura. Lo mejor de los
  dos mundos.
- Elegir bien estos parámetros es parte del oficio: notario para extraer y clasificar, poeta para
  escribir. La decisión es tuya.

**Para seguir:** el facsímil 4 (caja de herramientas) usa estos parámetros desde la API real; y el
facsímil 7 mide cuánto te puedes fiar de lo que sale, calibración incluida.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*